In [ ]:
import os
import sys
import subprocess
import warnings
from pathlib import Path

warnings.simplefilter("ignore")

WHEEL_DIRS = [
    "/kaggle/input/datasets/mayukh18/nemotron-packages/packages",
    "/kaggle/input/datasets/llkh0a/rtx-wheels/wheels",
]

OPTIONAL_WHEEL_FILES = [
    "/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    "/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
]

BASE_PKGS = [
    "sentencepiece",
    "safetensors",
    "huggingface_hub",
    "transformers",
    "jupyter_client",
    "pandas",
    "cbor2",
    "gguf",
    "msgspec",
    "pybase64",
    "openai_harmony",
    "llguidance",
    "diskcache",
    "xgrammar",
    "partial_json_parser",
    "model_hosting_container_standards",
    "prometheus_fastapi_instrumentator",
    "ijson",
    "mistral_common"
]

def pip_install_from_wheel_dirs(wheel_dirs, pkgs):
    for wd in wheel_dirs:
        if os.path.isdir(wd):
            cmd = [
                sys.executable, "-m", "pip", "install",
                "-q", "--no-index", "--find-links", wd, *pkgs
            ]
            print("Running:", " ".join(cmd))
            subprocess.run(cmd, check=False)
        else:
            print(f"колесо не нашлось, скип: {wd}")

def pip_install_optional_wheels(paths):
    for whl in paths:
        if os.path.exists(whl):
            cmd = [sys.executable, "-m", "pip", "install", "-q", whl]
            print("Running:", " ".join(cmd))
            subprocess.run(cmd, check=False)
        else:
            print(f"опшинел колесо не найдено, скип: {whl}")

pip_install_from_wheel_dirs(WHEEL_DIRS, BASE_PKGS)
pip_install_optional_wheels(OPTIONAL_WHEEL_FILES)
def pip_install_one_no_deps(wheel_dirs, pkg):
    for wd in wheel_dirs:
        if os.path.isdir(wd):
            cmd = [
                sys.executable, "-m", "pip", "install",
                "-q", "--no-index", "--find-links", wd,
                "--no-deps", pkg
            ]
            print("ранинг:", " ".join(cmd))
            subprocess.run(cmd, check=False)

pip_install_one_no_deps(WHEEL_DIRS, "vllm")

In [ ]:
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ.setdefault("TRITON_PTXAS_PATH", "/tmp/triton/backends/nvidia/bin/ptxas")

In [ ]:
import gc
import re
import time
import json
import math
import queue
import signal
import urllib.request
import urllib.error
import threading
import contextlib
from dataclasses import dataclass
from typing import Optional, List, Dict, Any
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from jupyter_client import KernelManager
from transformers import AutoTokenizer, set_seed
import kagglehub

In [ ]:
MODEL_CANDIDATE_PATHS = [
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default",
]
KAGGLEHUB_MODEL_HANDLE = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"

EVAL_CSV_PATHS = [
    "/kaggle/input/datasets/sohatiy/eval-set/vseros-aimo.csv",
    "/kaggle/input/datasets/sohatiy/eval-set2/reference.csv",
]
OUTPUT_DIR = "/kaggle/working/nemotron3nano30b_offline_tool_vs_notool"

QUESTION_CANDIDATES = ["question", "problem", "prompt", "input"]
ANSWER_CANDIDATES = ["answer", "target", "ground_truth", "gt", "label"]

N_PROBLEMS = 10
SAMPLES_PER_PROBLEM = 8

BASE_SEED = 42
SERVE_PORT = 8000
SERVED_MODEL_NAME = "model"

TEMPERATURE_NOTOOL = 1.0
TOP_P_NOTOOL = 1.0

TEMPERATURE_TOOL = 1.0
TOP_P_TOOL = 1.0

MAX_TURNS = 96
JUPYTER_TIMEOUT = 8
SANDBOX_ACQUIRE_TIMEOUT = 5
SERVER_TIMEOUT = 900

os.environ["VLLM_ENGINE_READY_TIMEOUT_S"] = "900"
os.environ["VLLM_NO_USAGE_STATS"] = "1"

GPU_MEMORY_UTILIZATION = 0.88
MAX_NUM_SEQS = 4
WORKERS = 4
ATTEMPT_TIMEOUT_SECONDS = 900

In [ ]:
SYSTEM_PROMPT = """
You are a careful mathematical reasoning model.

You are solving hard olympiad-style math problems.

Rules:
1. Work carefully and avoid arithmetic mistakes.
2. Explore, plan, execute, and verify.
3. The final answer must be a non-negative integer from 0 to 99999.
4. Put the final answer inside \\boxed{...}.
5. If you are unsure, still produce your best final boxed answer.

Be concise when possible. Do not ramble needlessly.
""".strip()

TOOL_INSTRUCTIONS = """
Use the python tool only when it meaningfully helps:
- exact arithmetic
- modular arithmetic
- brute force on small search spaces
- symbolic checks with sympy
- quick verification\n
Keep code short and use print().
Code should support reasoning, not replace it.
""".strip()

def build_system_prompt(use_tools: bool) -> str:
    if use_tools:
        return SYSTEM_PROMPT + "\n\n" + TOOL_INSTRUCTIONS
    return SYSTEM_PROMPT

In [ ]:
@dataclass
class ExperimentConfig:
    label: str
    use_tools: bool
    context_window: int
    generation_cap: int
    temperature: float
    top_p: float
    samples: int = SAMPLES_PER_PROBLEM

EXPERIMENTS = [
    ExperimentConfig(
        label="notool_32k",
        use_tools=False,
        context_window=32_768,
        generation_cap=32_768,
        temperature=TEMPERATURE_NOTOOL,
        top_p=TOP_P_NOTOOL,
    ),
    ExperimentConfig(
        label="tool_32k",
        use_tools=True,
        context_window=32_768,
        generation_cap=32_768,
        temperature=TEMPERATURE_TOOL,
        top_p=TOP_P_TOOL,
    ),
]

In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def resolve_model_path() -> str:
    def is_valid_model_dir(path: str) -> bool:
        return (
            os.path.isdir(path)
            and os.path.exists(os.path.join(path, "config.json"))
            and os.path.exists(os.path.join(path, "tokenizer.json"))
        )

    for p in MODEL_CANDIDATE_PATHS:
        if is_valid_model_dir(p):
            return p

    try:
        downloaded = kagglehub.model_download(KAGGLEHUB_MODEL_HANDLE)
        if is_valid_model_dir(downloaded):
            return downloaded

        for root, dirs, files in os.walk(downloaded):
            if {"config.json", "tokenizer.json"}.issubset(set(files)):
                return root
    except Exception as exc:
        print(f"kagglehub.model_download сломалось: {exc}")

MODEL_PATH = resolve_model_path()

def ensure_reasoning_parser(model_path: str, dst_path: str) -> str:
    for root, dirs, files in os.walk(model_path):
        if "nano_v3_reasoning_parser.py" in files:
            return os.path.join(root, "nano_v3_reasoning_parser.py")
    parser_code = '''
from vllm.reasoning.abs_reasoning_parsers import ReasoningParserManager
from vllm.reasoning.deepseek_r1_reasoning_parser import DeepSeekR1ReasoningParser

@ReasoningParserManager.register_module("nano_v3")
class NanoV3ReasoningParser(DeepSeekR1ReasoningParser):
    def extract_reasoning(self, model_output, request):
        reasoning_content, final_content = super().extract_reasoning(model_output, request)
        if (
            hasattr(request, "chat_template_kwargs")
            and request.chat_template_kwargs
            and request.chat_template_kwargs.get("enable_thinking") is False
            and final_content is None
        ):
            reasoning_content, final_content = final_content, reasoning_content
        return reasoning_content, final_content
'''.strip() + "\n"

    Path(dst_path).parent.mkdir(parents=True, exist_ok=True)
    with open(dst_path, "w", encoding="utf-8") as f:
        f.write(parser_code)
    return dst_path

REASONING_PARSER_PATH = ensure_reasoning_parser(
    MODEL_PATH,
    "/kaggle/working/nano_v3_reasoning_parser.py",
)

TOKENIZER = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True,
    use_fast=True,
)

def maybe_int(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    m = re.search(r"-?\d+", s)
    if not m:
        return None
    try:
        return int(m.group(0))
    except Exception:
        return None

def detect_question_answer_cols(df: pd.DataFrame):
    q_col = None
    a_col = None
    lower_map = {c.lower(): c for c in df.columns}

    for c in QUESTION_CANDIDATES:
        if c in lower_map:
            q_col = lower_map[c]
            break

    for c in ANSWER_CANDIDATES:
        if c in lower_map:
            a_col = lower_map[c]
            break

    return q_col, a_col

def load_eval_df() -> pd.DataFrame:
    def _load_one(path: str) -> pd.DataFrame:
        df = pd.read_csv(path)
        q_col, a_col = detect_question_answer_cols(df)
        lower_map = {c.lower(): c for c in df.columns}

        out = pd.DataFrame()

        if "problem_id" in lower_map:
            out["id"] = df[lower_map["problem_id"]]
        elif "id" in lower_map:
            out["id"] = df[lower_map["id"]]
        else:
            out["id"] = list(range(1, len(df) + 1))

        out["question"] = df[q_col].astype(str)

        dataset_tag = Path(path).stem
        out["id"] = out["id"].astype(str).map(lambda x: f"{dataset_tag}__{x}")

        if a_col is not None:
            out["gold_answer"] = df[a_col].apply(maybe_int)
        else:
            out["gold_answer"] = None

        return out

    if EVAL_CSV_PATHS:
        parts = []
        for path in EVAL_CSV_PATHS:
            part = _load_one(path).head(10).copy()
            parts.append(part)
        return pd.concat(parts, ignore_index=True)

def extract_answer(text: str) -> Optional[int]:
    if not text:
        return None

    patterns = [
        r"\\boxed\s*\{\s*([0-9,]+)\s*\}",
        r"boxed\s*\(\s*([0-9,]+)\s*\)",
        r"final\s+answer\s*(?:is|=|:)\s*([0-9,]+)",
        r"answer\s*(?:is|=|:)\s*([0-9,]+)",
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        if matches:
            try:
                v = int(matches[-1].replace(",", ""))
                if 0 <= v <= 99999:
                    return v
            except Exception:
                pass

    tail = text[-1500:]
    matches = re.findall(r"(?<!\d)([0-9]{1,5})(?!\d)", tail)
    if matches:
        try:
            v = int(matches[-1])
            if 0 <= v <= 99999:
                return v
        except Exception:
            pass

    return None

def write_trace_file(config_label: str, problem_id: str, sample_idx: int, payload: dict) -> str:
    safe_problem_id = re.sub(r"[^a-zA-Z0-9_-]", "_", str(problem_id))
    trace_dir = Path(OUTPUT_DIR) / config_label / "traces" / f"problem_{safe_problem_id}"
    trace_dir.mkdir(parents=True, exist_ok=True)
    path = trace_dir / f"sample_{int(sample_idx):02d}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    return str(path)

def estimate_prompt_tokens(
    messages: List[Dict[str, Any]],
    tools: Optional[List[Dict[str, Any]]] = None,
    enable_thinking: bool = True,
) -> Optional[int]:
    try:
        kwargs = dict(
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
        if tools is not None:
            kwargs["tools"] = tools

        tokenized = TOKENIZER.apply_chat_template(messages, **kwargs)
        if hasattr(tokenized, "tolist"):
            tokenized = tokenized.tolist()
        return len(tokenized)
    except Exception:
        return None

def safe_json_loads(s: str) -> Dict[str, Any]:
    try:
        return json.loads(s)
    except Exception:
        return {}

def normalize_content(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                if item.get("type") == "text":
                    parts.append(item.get("text", ""))
                elif "text" in item:
                    parts.append(str(item["text"]))
                else:
                    parts.append(str(item))
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content)

In [ ]:
def http_get_json(url: str, timeout: float = 30.0) -> dict:
    req = urllib.request.Request(
        url,
        headers={"Authorization": "Bearer sk-local"},
        method="GET",
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))

def http_post_json(url: str, payload: dict, timeout: float = ATTEMPT_TIMEOUT_SECONDS) -> dict:
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        url,
        data=data,
        headers={
            "Authorization": "Bearer sk-local",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))

In [ ]:
SANDBOX_BOOTSTRAP_CODE = r"""
import math
import numpy as np
import sympy as sp
import itertools
import collections
import mpmath
mpmath.mp.dps = 64

try:
    import pulp
    _OrigPULP_CBC_CMD = pulp.PULP_CBC_CMD

    class QuietPULP_CBC_CMD(_OrigPULP_CBC_CMD):
        def __init__(self, *args, **kwargs):
            kwargs.setdefault("msg", False)
            super().__init__(*args, **kwargs)

    pulp.PULP_CBC_CMD = QuietPULP_CBC_CMD

    try:
        pulp.LpSolverDefault.msg = False
    except Exception:
        pass
except Exception:
    pass
"""

class PythonSandbox:
    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout: float = JUPYTER_TIMEOUT):
        self.timeout = timeout
        self._km = None
        self._client = None
        self._owns_kernel = False

        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
        env["PYDEVD_WARN_EVALUATION_TIMEOUT"] = "0"
        env["JUPYTER_PLATFORM_DIRS"] = "1"
        env["PYTHONWARNINGS"] = "ignore"
        env["MPLBACKEND"] = "Agg"

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=["--Application.log_level=CRITICAL"])
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self.timeout)
        self._owns_kernel = True

        self.execute(SANDBOX_BOOTSTRAP_CODE)

    def _format_error(self, traceback: List[str]) -> str:
        clean_lines = []
        for frame in traceback:
            clean_frame = re.sub(r"\x1b\[[0-9;]*m", "", frame)
            if 'File "' in clean_frame and "ipython-input" not in clean_frame:
                continue
            clean_lines.append(clean_frame)
        return "".join(clean_lines)

    def execute(self, code: str, timeout: Optional[float] = None) -> str:
        effective_timeout = timeout or self.timeout
        client = self._client

        msg_id = client.execute(
            code,
            store_history=True,
            allow_stdin=False,
            stop_on_error=False,
        )

        stdout_parts = []
        stderr_parts = []
        start = time.time()

        while True:
            if time.time() - start > effective_timeout:
                self._km.interrupt_kernel()
                return f"[ERROR] Execution timed out after {effective_timeout} seconds"

            try:
                msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get("parent_header", {}).get("msg_id") != msg_id:
                continue

            msg_type = msg.get("msg_type")
            content = msg.get("content", {})

            if msg_type == "stream":
                text = content.get("text", "")
                if content.get("name") == "stdout":
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)

            elif msg_type == "error":
                stderr_parts.append(self._format_error(content.get("traceback", [])))

            elif msg_type in {"execute_result", "display_data"}:
                data = content.get("data", {})
                text = data.get("text/plain")
                if text:
                    stdout_parts.append(text if text.endswith("\n") else text + "\n")

            elif msg_type == "status" and content.get("execution_state") == "idle":
                break

        stdout = "".join(stdout_parts)
        stderr = "".join(stderr_parts)

        if stderr:
            return f"{stdout.rstrip()}\n{stderr}" if stdout else stderr

        return stdout if stdout.strip() else "[WARN] No output"

    def reset(self):
        self.execute("%reset -f\n" + SANDBOX_BOOTSTRAP_CODE)

    def close(self):
        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()
        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def __del__(self):
        self.close()

In [ ]:
PYTHON_EXEC_TOOL = [
    {
        "type": "function",
        "function": {
            "name": "stateful_python_code_exec",
            "description": (
                "Execute short Python code for mathematical reasoning, checking, "
                "exact arithmetic, modular arithmetic, small brute force, or sympy verification. "
                "Use print() for the key result."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "Python code to execute."
                    }
                },
                "required": ["code"],
            },
        },
    }
]

class PythonToolExecutor:
    def __init__(self, sandbox: PythonSandbox):
        self.sandbox = sandbox
        self.lock = threading.Lock()

    def run(self, arguments_json: str) -> str:
        args = safe_json_loads(arguments_json)
        code = args.get("code", "")

        if not isinstance(code, str) or not code.strip():
            return "[ERROR] Tool arguments must include a non-empty string field 'code'."

        with self.lock:
            try:
                return self.sandbox.execute(code)
            except Exception as exc:
                return f"[ERROR] {exc}"

In [ ]:
class NemotronRunner:
    def __init__(self, exp_cfg: ExperimentConfig, port: int = SERVE_PORT):
        self.exp_cfg = exp_cfg
        self.port = port
        self.base_url = f"http://127.0.0.1:{port}/v1"

        self.server_process = None
        self.log_file = None
        self.sandbox_pool = None

        self._start_server()
        self._wait_for_server()

        if exp_cfg.use_tools:
            self._init_sandboxes()

    def _start_server(self):
        log_path = os.path.join(OUTPUT_DIR, f"vllm_{self.exp_cfg.label}.log")
        self.log_file = open(log_path, "w")

        cmd = [
            sys.executable,
            "-m",
            "vllm.entrypoints.openai.api_server",
            "--model", MODEL_PATH,
            "--served-model-name", SERVED_MODEL_NAME,
            "--host", "0.0.0.0",
            "--port", str(self.port),
            "--tensor-parallel-size", "1",
            "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
            "--max-model-len", str(self.exp_cfg.context_window),
            "--max-num-seqs", str(MAX_NUM_SEQS),
            "--trust-remote-code",
            "--async-scheduling",
            "--kv-cache-dtype", "auto",
            "--reasoning-parser-plugin", REASONING_PARSER_PATH,
            "--reasoning-parser", "nano_v3",
            "--seed", str(BASE_SEED),
        ]

        if self.exp_cfg.use_tools:
            cmd += [
                "--enable-auto-tool-choice",
                "--tool-call-parser", "qwen3_coder",
            ]

        print("запуск vLLM:", " ".join(cmd))
        self.server_process = subprocess.Popen(
            cmd,
            stdout=self.log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    def _wait_for_server(self):
        start = time.time()
        models_url = f"{self.base_url}/models"

        for _ in range(SERVER_TIMEOUT):
            if self.server_process.poll() is not None:
                self.log_file.flush()
                with open(self.log_file.name, "r", encoding="utf-8", errors="ignore") as f:
                    logs = f.read()
                raise RuntimeError(f"vLLM server died.\n\n{logs}")

            try:
                _ = http_get_json(models_url, timeout=10)
                elapsed = time.time() - start
                print(f"[{self.exp_cfg.label}] vLLM ready in {elapsed:.1f}s")
                return
            except Exception:
                time.sleep(1)

        raise TimeoutError(f"[{self.exp_cfg.label}] vLLM server start timeout")

    def _init_sandboxes(self):
        self.sandbox_pool = queue.Queue()
        print(f"[{self.exp_cfg.label}] Initializing {WORKERS} sandboxes...")

        def create_one():
            return PythonSandbox(timeout=JUPYTER_TIMEOUT)

        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futures = [ex.submit(create_one) for _ in range(WORKERS)]
            for fut in as_completed(futures):
                self.sandbox_pool.put(fut.result())

    def _get_executor(self):
        if not self.exp_cfg.use_tools:
            return None, None
        sandbox = self.sandbox_pool.get(timeout=SANDBOX_ACQUIRE_TIMEOUT)
        return PythonToolExecutor(sandbox=sandbox), sandbox

    def _return_sandbox(self, sandbox):
        if sandbox is not None:
            try:
                sandbox.reset()
            except Exception:
                pass
            self.sandbox_pool.put(sandbox)

    def _base_messages(self, problem: str) -> List[Dict[str, Any]]:
        return [
            {"role": "system", "content": build_system_prompt(self.exp_cfg.use_tools)},
            {"role": "user", "content": problem},
        ]

    def _chat_once(
        self,
        messages: List[Dict[str, Any]],
        budget_remaining: int,
        seed: int,
    ):
        tools = PYTHON_EXEC_TOOL if self.exp_cfg.use_tools else None
        prompt_tokens_est = estimate_prompt_tokens(messages, tools=tools, enable_thinking=True)
        if prompt_tokens_est is None:
            prompt_tokens_est = 0

        available = self.exp_cfg.context_window - prompt_tokens_est - 128
        if available < 64 or budget_remaining <= 0:
            return None, prompt_tokens_est, available

        max_tokens = min(budget_remaining, available-1024)
        if max_tokens < 64:
            return None, prompt_tokens_est, available

        payload = {
            "model": SERVED_MODEL_NAME,
            "messages": messages,
            "temperature": self.exp_cfg.temperature,
            "top_p": self.exp_cfg.top_p,
            "max_tokens": max_tokens,
            "seed": seed,
            "chat_template_kwargs": {"enable_thinking": True},
        }

        if self.exp_cfg.use_tools:
            payload["tools"] = PYTHON_EXEC_TOOL
            payload["tool_choice"] = "auto"

        response = http_post_json(
            f"{self.base_url}/chat/completions",
            payload,
            timeout=ATTEMPT_TIMEOUT_SECONDS,
        )
        return response, prompt_tokens_est, available

    def run_sample(
        self,
        problem_id: str,
        problem: str,
        gold_answer: Optional[int],
        sample_idx: int,
    ) -> Dict[str, Any]:
        deadline = time.time() + ATTEMPT_TIMEOUT_SECONDS
        seed = BASE_SEED + (abs(hash(str(problem_id))) % 1_000_000) + int(sample_idx)

        executor = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        turns_taken = 0
        final_answer = None
        full_text = ""
        hit_timeout = False
        hit_context_limit = False
        hit_generation_cap = False

        budget_remaining = self.exp_cfg.generation_cap
        trace_events = []

        try:
            executor, sandbox = self._get_executor()
            messages = self._base_messages(problem)

            trace_events.append({
                "event": "metadata",
                "config": self.exp_cfg.label,
                "problem_id": str(problem_id),
                "sample_idx": int(sample_idx),
                "seed": int(seed),
                "use_tools": bool(self.exp_cfg.use_tools),
                "context_window": int(self.exp_cfg.context_window),
                "generation_cap": int(self.exp_cfg.generation_cap),
                "temperature": float(self.exp_cfg.temperature),
                "top_p": float(self.exp_cfg.top_p),
            })
            trace_events.append({
                "event": "input",
                "problem": problem,
                "gold_answer": gold_answer,
            })

            for turn_idx in range(1, MAX_TURNS + 1):
                turns_taken = turn_idx

                if time.time() > deadline:
                    hit_timeout = True
                    trace_events.append({
                        "event": "stop",
                        "reason": "timeout",
                        "turn": turn_idx,
                    })
                    break

                response, prompt_tokens_est, available = self._chat_once(messages, budget_remaining, seed)

                if response is None:
                    if budget_remaining <= 0:
                        hit_generation_cap = True
                        stop_reason = "generation_cap"
                    else:
                        hit_context_limit = True
                        stop_reason = "context_limit"

                    trace_events.append({
                        "event": "stop",
                        "reason": stop_reason,
                        "turn": turn_idx,
                        "prompt_tokens_est": prompt_tokens_est,
                        "available": available,
                        "budget_remaining": budget_remaining,
                    })
                    break

                choice = response["choices"][0]
                assistant_msg = choice["message"]
                usage = response.get("usage", {})

                generated_this_turn = int(usage.get("completion_tokens", 0) or 0)
                prompt_tokens_api = int(usage.get("prompt_tokens", 0) or 0)
                total_tokens += generated_this_turn
                budget_remaining -= generated_this_turn

                content = normalize_content(assistant_msg.get("content"))
                reasoning_content = normalize_content(assistant_msg.get("reasoning_content"))
                if reasoning_content:
                    full_text += reasoning_content + "\n"
                if content:
                    full_text += content

                tool_calls = []
                raw_tool_calls = assistant_msg.get("tool_calls") or []
                if raw_tool_calls:
                    for tc in raw_tool_calls:
                        fn = tc.get("function", {})
                        tool_calls.append({
                            "id": tc.get("id"),
                            "type": "function",
                            "function": {
                                "name": fn.get("name"),
                                "arguments": fn.get("arguments", ""),
                            },
                        })

                trace_events.append({
                    "event": "assistant_response",
                    "turn": turn_idx,
                    "prompt_tokens_est": prompt_tokens_est,
                    "prompt_tokens_api": prompt_tokens_api,
                    "available_before_turn": available,
                    "generated_tokens": generated_this_turn,
                    "budget_remaining_after_turn": budget_remaining,
                    "reasoning_content": reasoning_content,
                    "content": content,
                    "tool_calls": tool_calls,
                    "finish_reason": choice.get("finish_reason"),
                })

                finish_reason = choice.get("finish_reason")
                if finish_reason == "length":
                    hit_generation_cap = True

                assistant_payload = {
                    "role": "assistant",
                    "content": content,
                }
                if tool_calls:
                    assistant_payload["tool_calls"] = tool_calls
                messages.append(assistant_payload)

                maybe_answer = extract_answer(content) or extract_answer(full_text)
                if maybe_answer is not None:
                    final_answer = maybe_answer

                if not tool_calls:
                    trace_events.append({
                        "event": "stop",
                        "reason": "assistant_final",
                        "turn": turn_idx,
                    })
                    break

                for tc in tool_calls:
                    python_calls += 1
                    name = tc["function"]["name"]
                    arguments = tc["function"]["arguments"]

                    if name != "stateful_python_code_exec":
                        tool_output = f"[ERROR] Unknown tool name: {name}"
                        python_errors += 1
                    else:
                        tool_output = executor.run(arguments)
                        if tool_output.startswith("[ERROR]") or "Traceback" in tool_output:
                            python_errors += 1

                    trace_events.append({
                        "event": "tool_request",
                        "turn": turn_idx,
                        "tool": name,
                        "arguments": arguments,
                    })
                    trace_events.append({
                        "event": "tool_response",
                        "turn": turn_idx,
                        "tool": name,
                        "output": tool_output,
                    })

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc["id"],
                        "content": tool_output,
                    })

                if budget_remaining <= 0:
                    hit_generation_cap = True
                    trace_events.append({
                        "event": "stop",
                        "reason": "generation_cap",
                        "turn": turn_idx,
                        "budget_remaining": budget_remaining,
                    })
                    break

            if final_answer is None:
                final_answer = extract_answer(full_text)

        except Exception as exc:
            python_errors += 1
            full_text += f"\n[RUN_ERROR] {exc}"
            trace_events.append({
                "event": "run_error",
                "error": str(exc),
            })

        finally:
            self._return_sandbox(sandbox)

        is_correct = None
        if gold_answer is not None and final_answer is not None:
            is_correct = int(final_answer == gold_answer)

        trace_payload = {
            "config": self.exp_cfg.label,
            "problem_id": str(problem_id),
            "sample_idx": int(sample_idx),
            "gold_answer": gold_answer,
            "pred_answer": final_answer,
            "is_correct": is_correct,
            "total_tokens": int(total_tokens),
            "python_calls": int(python_calls),
            "python_errors": int(python_errors),
            "turns_taken": int(turns_taken),
            "hit_timeout": bool(hit_timeout),
            "hit_context_limit": bool(hit_context_limit),
            "hit_generation_cap": bool(hit_generation_cap),
            "full_text": full_text,
            "events": trace_events,
        }

        trace_path = write_trace_file(
            config_label=self.exp_cfg.label,
            problem_id=problem_id,
            sample_idx=int(sample_idx),
            payload=trace_payload,
        )

        return {
            "config": self.exp_cfg.label,
            "problem_id": str(problem_id),
            "sample_idx": int(sample_idx),
            "gold_answer": gold_answer,
            "pred_answer": final_answer,
            "is_correct": is_correct,
            "total_tokens": int(total_tokens),
            "python_calls": int(python_calls),
            "python_errors": int(python_errors),
            "turns_taken": int(turns_taken),
            "hit_timeout": int(hit_timeout),
            "hit_context_limit": int(hit_context_limit),
            "hit_generation_cap": int(hit_generation_cap),
            "trace_path": trace_path,
            "raw_tail": full_text[-3000:],
        }

    def run_problem(self, row: pd.Series):
        problem_id = str(row["id"])
        question = str(row["question"])
        gold_answer = maybe_int(row["gold_answer"]) if "gold_answer" in row else None

        print(f"\n[{self.exp_cfg.label}] задача {problem_id}")

        sample_rows = []
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futures = [
                ex.submit(self.run_sample, problem_id, question, gold_answer, i)
                for i in range(self.exp_cfg.samples)
            ]
            for fut in as_completed(futures):
                sample_rows.append(fut.result())

        sample_rows = sorted(sample_rows, key=lambda x: x["sample_idx"])

        answers = [r["pred_answer"] for r in sample_rows if r["pred_answer"] is not None]
        maj_answer = None
        maj_votes = 0

        if answers:
            cnt = Counter(answers).most_common()
            maj_answer, maj_votes = cnt[0]

        problem_correct = None
        if gold_answer is not None and maj_answer is not None:
            problem_correct = int(maj_answer == gold_answer)

        problem_row = {
            "config": self.exp_cfg.label,
            "problem_id": problem_id,
            "gold_answer": gold_answer,
            "majority_answer": maj_answer,
            "majority_votes": maj_votes,
            "problem_correct": problem_correct,
            "num_valid_answers": len(answers),
            "mean_tokens": float(pd.Series([r["total_tokens"] for r in sample_rows]).mean()),
            "median_tokens": float(pd.Series([r["total_tokens"] for r in sample_rows]).median()),
            "p90_tokens": float(pd.Series([r["total_tokens"] for r in sample_rows]).quantile(0.90)),
            "mean_python_calls": float(pd.Series([r["python_calls"] for r in sample_rows]).mean()),
            "mean_python_errors": float(pd.Series([r["python_errors"] for r in sample_rows]).mean()),
            "timeouts": int(sum(r["hit_timeout"] for r in sample_rows)),
            "context_hits": int(sum(r["hit_context_limit"] for r in sample_rows)),
            "cap_hits": int(sum(r["hit_generation_cap"] for r in sample_rows)),
        }

        return problem_row, sample_rows

    def shutdown(self):
        if self.server_process is not None:
            with contextlib.suppress(Exception):
                os.killpg(os.getpgid(self.server_process.pid), signal.SIGTERM)
            with contextlib.suppress(Exception):
                self.server_process.wait(timeout=20)

        if self.log_file is not None:
            with contextlib.suppress(Exception):
                self.log_file.close()

        if self.sandbox_pool is not None:
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
                except Exception:
                    pass

    def __del__(self):
        self.shutdown()

In [ ]:
def run_experiment(exp_cfg: ExperimentConfig, eval_df: pd.DataFrame):
    print("\n" + "=" * 80)
    print(f"запуск: {exp_cfg.label}")
    print("=" * 80)

    problems_csv = os.path.join(OUTPUT_DIR, f"{exp_cfg.label}_problems.csv")
    samples_csv = os.path.join(OUTPUT_DIR, f"{exp_cfg.label}_samples.csv")
    samples_jsonl = os.path.join(OUTPUT_DIR, f"{exp_cfg.label}_samples.jsonl")

    for p in [problems_csv, samples_csv, samples_jsonl]:
        if os.path.exists(p):
            os.remove(p)

    runner = NemotronRunner(exp_cfg)
    all_problem_rows = []
    all_sample_rows = []

    try:
        for _, row in eval_df.iterrows():
            gc.collect()
            problem_row, sample_rows = runner.run_problem(row)
            all_problem_rows.append(problem_row)
            all_sample_rows.extend(sample_rows)

            pd.DataFrame(all_problem_rows).to_csv(problems_csv, index=False)
            pd.DataFrame(all_sample_rows).to_csv(samples_csv, index=False)

            with open(samples_jsonl, "a", encoding="utf-8") as f:
                for r in sample_rows:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")

    finally:
        runner.shutdown()
        gc.collect()

    problems_df = pd.DataFrame(all_problem_rows)
    samples_df = pd.DataFrame(all_sample_rows)

    summary = {
        "config": exp_cfg.label,
        "n_problems": len(problems_df),
        "n_samples": len(samples_df),
        "mean_problem_tokens": problems_df["mean_tokens"].mean() if len(problems_df) else None,
        "mean_sample_tokens": samples_df["total_tokens"].mean() if len(samples_df) else None,
        "mean_python_calls": samples_df["python_calls"].mean() if len(samples_df) else None,
        "sample_valid_rate": samples_df["pred_answer"].notna().mean() if len(samples_df) else None,
        "cap_hit_rate": samples_df["hit_generation_cap"].mean() if len(samples_df) else None,
        "context_hit_rate": samples_df["hit_context_limit"].mean() if len(samples_df) else None,
    }

    if "problem_correct" in problems_df.columns and problems_df["problem_correct"].notna().any():
        summary["problem_accuracy"] = problems_df["problem_correct"].mean()

    if "is_correct" in samples_df.columns and samples_df["is_correct"].notna().any():
        summary["sample_accuracy"] = samples_df["is_correct"].mean()

    return problems_df, samples_df, summary

In [ ]:
set_seed(BASE_SEED)

eval_df = load_eval_df()
all_summaries = []

for exp_cfg in EXPERIMENTS:
    problems_df, samples_df, summary = run_experiment(exp_cfg, eval_df)
    all_summaries.append(summary)

summary_df = pd.DataFrame(all_summaries)
summary_path = os.path.join(OUTPUT_DIR, "summary.csv")
summary_df.to_csv(summary_path, index=False)